# **Label-free CBM and LaBo: a concept bottleneck without concept labels**

Practice for the module [“Concept-based methods”](https://ai-interpretability.school), lesson “Concept Bottleneck Models (CBM)”, steps “Concepts without manual labeling: Label-free CBM” and “Concepts without manual labeling: LaBo”.

In the notebook `PRACTICE_cbm_cem` the concepts of XOR and dot data were labeled in advance, and the CBM learned to predict them. Real images usually come without such labels. Label-free CBM ([Oikarinen et al., ICLR 2023](https://arxiv.org/abs/2304.06129)) and LaBo ([Yang et al., CVPR 2023](https://arxiv.org/abs/2211.11158)) replace them with two ready-made models: a language model writes the list of concepts, and CLIP decides how visible a concept is in an image. Here we build both methods with our own `torch` code on top of CLIP ViT-B/32 and CIFAR-10:

1. data: a CIFAR-10 subset, features of the ResNet-18 backbone and CLIP embeddings;
2. the GPT-3 concept list from the Label-free CBM repository and the concept filters;
3. the Label-free CBM concept layer: a projection of ResNet-18 features trained to reproduce CLIP similarities;
4. a sparse last layer with elastic net and its sensitivity to the sparsity strength;
5. LaBo: concept selection for each class and the “classes × concepts” weight matrix;
6. a comparison of accuracy and the number of nonzero last-layer weights with the CBM from `PRACTICE_cbm_cem`, CLIP without training and linear classifiers over features;
7. where the methods break.

The concepts come as a ready file; GPT-3 is not called. Both models (ResNet-18, about 45 MB, and CLIP, about 600 MB) and CIFAR-10 (about 170 MB) are downloaded on the first run without keys. With a GPU (Runtime → Change runtime type → T4 GPU) encoding 7000 images takes about a minute, on the Colab CPU about 10 minutes; the rest of the notebook runs in a couple of minutes.

In [ ]:
!pip install -q transformers==5.18.0

In [ ]:
import json
import urllib.request
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm without ipywidgets; Colab has ipywidgets
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import transformers
from torchvision import transforms
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
print(device, torch.__version__, transformers.__version__)

## 1. Data

CIFAR-10 has 10 classes and 32×32 images. To keep the notebook runnable on CPU, we take a subset with a fixed seed: 400 images of each class for training, another 100 of each class from the training part for validation and 200 of each class from the test part, 4000, 1000 and 2000 images in total.

Each image goes through two networks:

- **the backbone** is ResNet-18 trained on ImageNet, without its last layer: 512 features. Label-free CBM projects its features into concepts; CLIP takes no part in it;
- **CLIP ViT-B/32** gives a normalized image embedding, also 512 numbers. The “image — concept text” similarities are computed with it.

For CIFAR-10 the Label-free CBM paper also uses different networks as the backbone and as CLIP. If CLIP itself were the backbone, the concept layer would be trivial: the similarity to a text is a linear function of the same embedding.

In [ ]:
train_set = torchvision.datasets.CIFAR10('data', train=True, download=True)
test_set = torchvision.datasets.CIFAR10('data', train=False, download=True)
CLASSES = train_set.classes
rng = np.random.default_rng(SEED)


def per_class(targets, n):
    # n random indices of each class
    y = np.array(targets)
    return [rng.permutation(np.where(y == k)[0])[:n] for k in range(10)]


blocks = per_class(train_set.targets, 500)
idx = {'train': np.concatenate([b[:400] for b in blocks]),
       'val': np.concatenate([b[400:] for b in blocks]),
       'test': np.concatenate(per_class(test_set.targets, 200))}
source = {'train': train_set, 'val': train_set, 'test': test_set}
y = {s: torch.tensor(np.array(source[s].targets)[idx[s]], device=device) for s in idx}
print(CLASSES)
print({s: len(idx[s]) for s in idx})

In [ ]:
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
tokenizer = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
resnet.fc = torch.nn.Identity()          # 512 penultimate features
resnet_prep = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                                  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


@torch.no_grad()
def encode_images(split, batch=250):
    ds, ids = source[split], idx[split]
    clip_emb, res_feat = [], []
    for s in range(0, len(ids), batch):
        images = [ds[i][0] for i in ids[s:s + batch]]
        pixels = processor(images=images, return_tensors='pt')['pixel_values'].to(device)
        clip_emb.append(F.normalize(clip.get_image_features(pixel_values=pixels).pooler_output, dim=-1))
        res_feat.append(resnet(torch.stack([resnet_prep(im) for im in images]).to(device)))
    return torch.cat(clip_emb), torch.cat(res_feat)


@torch.no_grad()
def encode_text(texts):
    tokens = tokenizer(texts, padding=True, return_tensors='pt').to(device)
    return F.normalize(clip.get_text_features(**tokens).pooler_output, dim=-1)


X, B = {}, {}                            # X are CLIP embeddings, B are ResNet-18 features
for s in idx:
    X[s], B[s] = encode_images(s)
mu, sd = B['train'].mean(0), B['train'].std(0)
B = {s: (B[s] - mu) / sd for s in B}     # standardized with training statistics
print({s: tuple(X[s].shape) for s in X}, {s: tuple(B[s].shape) for s in B})

Two reference points that do not explain their prediction with concepts: CLIP without training (zero-shot: the class whose text `a photo of a {class}` is closest to the image) and linear classifiers over ResNet-18 features and over the CLIP embedding. The classifier is trained by the same function `fit_linear` that is later used for the last layer over concepts: full batch, cross-entropy, gradient descent.

In [ ]:
def fit_linear(Z, labels, steps=2000, lr=0.1):
    torch.manual_seed(SEED)
    W = torch.zeros(10, Z.shape[1], device=device, requires_grad=True)
    b = torch.zeros(10, device=device, requires_grad=True)
    opt = torch.optim.SGD([W, b], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        F.cross_entropy(Z @ W.T + b, labels).backward()
        opt.step()
    return W.detach(), b.detach()


def accuracy(Z, W, b, labels):
    return ((Z @ W.T + b).argmax(1) == labels).float().mean().item()


T_prompt = encode_text([f'a photo of a {c}' for c in CLASSES])
results = {}
results['CLIP zero-shot'] = (((X['test'] @ T_prompt.T).argmax(1) == y['test']).float().mean().item(), 0)
W, b = fit_linear(B['train'], y['train'])
results['linear classifier over ResNet-18'] = (accuracy(B['test'], W, b, y['test']), int((W != 0).sum()))
W, b = fit_linear(10 * X['train'], y['train'])   # ×10: the CLIP embedding is normalized, without the factor descent is slow
results['linear classifier over CLIP'] = (accuracy(10 * X['test'], W, b, y['test']), int((W != 0).sum()))
for k, (a, n) in results.items():
    print(f'{k:45s} accuracy {a:.4f}   nonzero weights {n}')

## 2. The concept list and the filters

For CIFAR-10 the Label-free CBM authors asked GPT-3 three questions about each class and published the answers in their repository: `important` — which features identify the class, `around` — what is most often seen next to it, `superclass` — which more general categories the class belongs to. We take these three files as they are.

In [ ]:
URL = 'https://raw.githubusercontent.com/Trustworthy-ML-Lab/Label-free-CBM/main/data/concept_sets/gpt3_init/gpt3_cifar10_{}.json'
gpt = {q: json.loads(urllib.request.urlopen(URL.format(q), timeout=30).read()) for q in ('important', 'around', 'superclass')}
for q in gpt:
    print(q, '|', 'airplane:', gpt[q]['airplane'])

**Task 1. Build the dictionary `pools`: for each class, a sorted list of its concepts from the three files without repeats, strings lowercased and stripped of surrounding spaces. Build `concepts`, a sorted list of all concepts of all classes without repeats. How many lines do the three files hold together, and how many distinct concepts are in `concepts`?**

In [ ]:
# Your code here

Label-free CBM cleans the list with four filters, three of which look only at the text:

1. **length**: a concept longer than 30 characters is removed, it is hard to read in an explanation;
2. **similarity to a class name**: a concept that nearly repeats a class name (`a cat` for the class `cat`) is removed, otherwise the bottleneck passes the answer itself;
3. **similarity to another concept**: of a pair of near-identical concepts (`a bird feeder` and `a birdfeeder`) one is kept.

Similarity here is the cosine between CLIP text embeddings. The authors also compute it with a second language model (all-mpnet-base-v2) and use the cutoffs 0.85 and 0.9; we use CLIP alone, and its embeddings of short texts of the form `a ...` are close to each other almost always, so the cutoffs are higher: 0.90 for class names and 0.95 for concept pairs.

**Task 2. Write the function `text_filters(concepts, class_cut=0.90, pair_cut=0.95)`. It removes concepts longer than 30 characters; then concepts whose largest cosine with the embedding of any class name (`encode_text(CLASSES)`) exceeds `class_cut`; then it walks through the rest in the order “shorter first, alphabetical within one length” and keeps a concept only if its cosine with every concept already kept is below `pair_cut`. The function prints the number of concepts after each filter and returns the list of kept ones. How many concepts remain after the three filters?**

In [ ]:
# Your code here

The fourth filter looks at images: **presence**. For each concept one takes the five training images with the largest CLIP cosine “image embedding — text embedding” and averages these five numbers. If the mean is not above 0.25 (the authors' cutoff), CLIP hardly finds the concept in any image, and the concept is removed.

In [ ]:
def presence_filter(names, cut=0.25, verbose=True):
    top5 = (X['train'] @ encode_text(names).T).topk(5, dim=0).values.mean(0)
    if verbose:
        print('removed:', [n for n, t in zip(names, top5) if t <= cut])
    return [n for n, t in zip(names, top5) if t > cut]


after_presence = presence_filter(after_text)
print('left:', len(after_presence))

## 3. The Label-free CBM concept layer

Let $f(x)\in\mathbb{R}^{512}$ be the ResNet-18 features and $P_{ij}$ the CLIP cosine between image $i$ and concept $j$. The concept layer is a linear projection $W_c\in\mathbb{R}^{K\times 512}$ (here $K$ is the number of concepts, which the lesson denotes $M$, while $K$ there is the number of classes), trained so that the column of activations of concept $j$ over all images, $q_j = (W_c f(x_i))_{i}$, reproduces the column $P_{:,j}$ up to shift and scale. The measure is the cosine of cubes:

$$\mathrm{sim}(q_j, P_{:,j}) = \cos\big(\overline{q_j^{\,3}},\ \overline{P_{:,j}^{\,3}}\big),$$

where the cube is elementwise and the bar means “subtract the mean over images”. The cube amplifies the largest values: what matters most is that the concept activation is high on the images where CLIP sees the concept best. The loss is $-\frac{1}{K}\sum_j \mathrm{sim}(q_j, P_{:,j})$ on training images.

**Task 3. Write the function `cos_cubed(a, b)`: for matrices $a, b$ of size “images × concepts” it returns a vector of length $K$, the cosine between the columns of $a^3$ and $b^3$ with the mean over images subtracted.**

In [ ]:
# Your code here

We train the projection with Adam on the full batch for up to 1000 steps and keep the weights with the best mean $\mathrm{sim}$ on validation: on 4000 images a 512 → $K$ projection overfits quickly. The last filter is **interpretability**: a concept whose validation $\mathrm{sim}$ is not above 0.45 (the authors' cutoff) is reproduced poorly by the concept layer and is removed. Then the activations of the remaining concepts are standardized with training statistics: this is the bottleneck $c(x)$.

In [ ]:
def concept_layer(names, steps=1000, lr=1e-3, cut=0.45, verbose=True):
    T = encode_text(names)
    P_train, P_val = X['train'] @ T.T, X['val'] @ T.T
    torch.manual_seed(SEED)
    Wc = torch.nn.Linear(512, len(names), bias=False).to(device)
    opt = torch.optim.Adam(Wc.parameters(), lr=lr)
    best, best_W, best_step = -1.0, None, 0
    for step in range(1, steps + 1):
        opt.zero_grad()
        (-cos_cubed(Wc(B['train']), P_train).mean()).backward()
        opt.step()
        if step % 50 == 0:
            with torch.no_grad():
                v = cos_cubed(Wc(B['val']), P_val).mean().item()
            if v > best:
                best, best_W, best_step = v, Wc.weight.detach().clone(), step
    with torch.no_grad():
        sim_val = cos_cubed(B['val'] @ best_W.T, P_val)
    good = sim_val > cut
    if verbose:
        print(f'best step {best_step}, mean validation sim {best:.3f}')
        print('removed by the interpretability filter:', [n for n, g in zip(names, good) if not g])
    W_keep = best_W[good]
    C = {s: B[s] @ W_keep.T for s in B}
    m, s_ = C['train'].mean(0), C['train'].std(0)
    return [n for n, g in zip(names, good) if g], {s: (C[s] - m) / s_ for s in C}, sim_val[good]


lf_names, C, lf_sim = concept_layer(after_presence)
print('concepts in the bottleneck:', len(lf_names))

## 4. The sparse last layer

The classes are predicted by a linear layer $\hat y = \arg\max_k (W c(x) + b)_k$, trained with elastic net:

$$\min_{W,b}\ \mathrm{CE}(Wc+b,\ y) + \lambda\Big(\alpha\,\|W\|_1 + \frac{1-\alpha}{2}\,\|W\|_2^2\Big),\qquad \alpha=0.99.$$

The term $\|W\|_1$ is not differentiable at zero, so after an ordinary gradient step on the smooth part one makes a **proximal step**: every weight is moved towards zero by $\eta\lambda\alpha$ ($\eta$ is the learning rate), and a weight that would cross zero is set to zero: $w \leftarrow \mathrm{sign}(w)\max(|w|-\eta\lambda\alpha,\ 0)$. This makes weights exactly zero rather than just small. The authors solve the same problem with the GLM-SAGA package; a full batch on 4000 images is simpler to do by hand.

**Task 4. Write the function `fit_sparse(Z, labels, lam, alpha=0.99, steps=2000, lr=0.1)`: weights and bias start at zero, full batch, at each step the gradient of cross-entropy plus $\lambda\frac{1-\alpha}{2}\|W\|_2^2$, a descent step for $W$ and $b$ and a proximal step for $W$. It returns `W, b`. Train it on `C['train']` for each $\lambda$ in the list `LAMS` and fill the table: test accuracy and the number of nonzero weights.**

In [ ]:
# Your code here

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep.iloc[:, 2], sweep.iloc[:, 1], 'o-', color='#2a6f97')
for _, r in sweep.iterrows():
    ax.annotate(f"λ={r['lambda']}", (r.iloc[2], r.iloc[1]), textcoords='offset points', xytext=(5, -12), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('nonzero weights of the last layer (log scale)')
ax.set_ylabel('test accuracy')
ax.set_title('Label-free CBM: accuracy against sparsity')
ax.grid(alpha=0.3)
plt.show()

From here on we take $\lambda=0.003$: each class keeps about two dozen nonzero weights, and in the solution run the test accuracy is about 2 points below that of the layer without sparsity. Let us see which concepts each class relies on: the three concepts with the largest positive weights.

In [ ]:
LAM = 0.003
W_lf, b_lf = fit_sparse(C['train'], y['train'], LAM)
results['Label-free CBM, λ=0'] = (accuracy(C['test'], *fit_sparse(C['train'], y['train'], 0.0), y['test']), len(lf_names) * 10)
results[f'Label-free CBM, λ={LAM}'] = (accuracy(C['test'], W_lf, b_lf, y['test']), int((W_lf != 0).sum()))


def top_concepts(W, names, k=3):
    return pd.DataFrame({c: [f'{names[j]} ({W[i, j]:.2f})' for j in W[i].argsort(descending=True)[:k].tolist()]
                         for i, c in enumerate(CLASSES)}).T


top_concepts(W_lf, lf_names)

The explanation of one prediction is the concept contributions to the logit of the predicted class: $W_{\hat y j}\,c_j(x)$. A sparse layer has about two dozen nonzero terms, and the eight largest in absolute value already show what the logit consists of.

In [ ]:
i = 7                                    # index of the image in the test subset
logits = C['test'][i] @ W_lf.T + b_lf
pred = logits.argmax().item()
contrib = W_lf[pred] * C['test'][i]
order = contrib.abs().argsort(descending=True)[:8].tolist()[::-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={'width_ratios': [1, 2.4]})
axes[0].imshow(test_set[idx['test'][i]][0])
axes[0].set_title(f'true: {CLASSES[y["test"][i]]}, predicted: {CLASSES[pred]}', fontsize=9)
axes[0].axis('off')
axes[1].barh([lf_names[j] for j in order], contrib[order].cpu(), color=['#2a6f97' if v > 0 else '#c0392b' for v in contrib[order]])
axes[1].set_xlabel(f'contribution W[{CLASSES[pred]}, j] · c_j(x) to the logit')
axes[1].set_title('eight largest concept contributions', fontsize=9)
fig.tight_layout()
plt.show()

A bottleneck concept's value is the similarity CLIP computes. This can only be checked by eye: we show the six test images with the largest activation of the two concepts that the classes `horse` and `ship` rely on most.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for row, cls in enumerate(('horse', 'ship')):
    j = W_lf[CLASSES.index(cls)].argmax().item()
    for col, t in enumerate(C['test'][:, j].argsort(descending=True)[:6].tolist()):
        axes[row, col].imshow(test_set[idx['test'][t]][0])
        axes[row, col].set_title(CLASSES[y['test'][t]], fontsize=8)
        axes[row, col].axis('off')
    axes[row, 0].text(-4, 16, lf_names[j], ha='right', va='center', fontsize=9)
fig.suptitle('six test images with the largest activation of the concept (title: true class)', fontsize=10)
fig.tight_layout()
plt.show()

## 5. LaBo

LaBo is built differently. The bottleneck is not trained: the value of concept $j$ on an image is the CLIP cosine $g_j(x) = \langle e_{\text{img}}(x), e_{\text{text}}(j)\rangle$. Only the “classes × concepts” matrix is trained, and at the start of training each concept is linked only to its own class.

**Concept selection.** For each class, $k$ of its concepts are chosen so that they both separate the class from the others and do not repeat each other. The authors greedily maximize a submodular function of two terms:

- **discriminability** $D(j)$: the mean cosine with concept $j$ over the images of the class is compared with the same means of the other classes; here it is the share of the class in a softmax over the ten classes with temperature 0.01;
- **coverage**: for each concept of the pool one takes its largest cosine with the concepts already chosen and averages over the pool. This term grows more slowly if a new concept resembles the ones already chosen.

At each step one adds the concept that increases the sum $\sum_{j\in S} D(j) + \text{coverage}(S)$ the most. The pool of a class is its concepts from the three files with those that nearly repeat the class name removed (the same 0.90 filter). For comparison, the same size $k$ is also drawn at random.

In [ ]:
class_means = torch.stack([X['train'][y['train'] == k].mean(0) for k in range(10)])
labo_pool = {}
for c in CLASSES:
    T = encode_text(pools[c])
    keep = (T @ T_class.T).max(1).values <= 0.90
    labo_pool[c] = ([n for n, kk in zip(pools[c], keep) if kk], T[keep])
print({c: len(labo_pool[c][0]) for c in CLASSES})


def select_concepts(k, random=False):
    names, embs, owner = [], [], []
    g = torch.Generator().manual_seed(SEED)
    for ci, c in enumerate(CLASSES):
        pool, T = labo_pool[c]
        if random:
            chosen = torch.randperm(len(pool), generator=g)[:k].tolist()
        else:
            D = torch.softmax((class_means @ T.T) / 0.01, dim=0)[ci]
            sim = T @ T.T
            chosen = []
            for _ in range(min(k, len(pool))):
                gain = [D[chosen + [j]].sum() + sim[:, chosen + [j]].max(1).values.mean() if j not in chosen else -1e9
                        for j in range(len(pool))]
                chosen.append(int(torch.tensor(gain).argmax()))
        names += [pool[j] for j in chosen]
        embs.append(T[chosen])
        owner += [ci] * len(chosen)
    return names, torch.cat(embs), torch.tensor(owner, device=device)


labo_names, labo_T, labo_owner = select_concepts(5)
pd.DataFrame({c: labo_names[5 * i:5 * i + 5] for i, c in enumerate(CLASSES)}).T

**The weight matrix.** Let $g(x)\in\mathbb{R}^{K}$ be the cosines of the image with the selected concepts, standardized with training statistics. The weights are a matrix $A\in\mathbb{R}^{10\times K}$ whose every row is a distribution over concepts: $A = \mathrm{softmax}(V)$ row-wise, where $V$ is trained. The logit of a class is $\tau\,\langle A_k, g(x)\rangle$ with the factor $\tau=10$. The language prior enters through the start of training: $V_{kj} = 5$ if concept $j$ was selected for class $k$, and 0 otherwise.

**Task 5. Write the function `fit_labo(T, owner, steps=500, lr=0.01)`: from the text embeddings of the selected concepts `T` and the indices of their classes `owner` it builds $g$ for the training and test parts, trains $V$ with Adam on the full batch with cross-entropy and returns the test accuracy and the matrix $A$. Compute the accuracy of LaBo with $k=1, 3, 5, 10$ concepts per class under submodular and random selection.**

In [ ]:
# Your code here

The matrix $A$ after training: what share of each class's weight stayed on its own concepts. At the start, $V_{kj}=5$ gives each own concept the weight $e^5/(5e^5+45)\approx 0.19$, and the five own concepts carry about 0.94 of the row weight.

In [ ]:
acc_labo, A_labo = fit_labo(labo_T, labo_owner)
results['LaBo, 5 concepts per class'] = (acc_labo, int((A_labo != 0).sum()))
own = (A_labo * F.one_hot(labo_owner, 10).T).sum(1)
print('share of weight on own concepts per class:', {c: round(v, 3) for c, v in zip(CLASSES, own.tolist())})

fig, ax = plt.subplots(figsize=(11, 3.4))
im = ax.imshow(A_labo.cpu(), aspect='auto', cmap='Blues')
ax.set_yticks(range(10), CLASSES, fontsize=8)
ax.set_xticks(range(0, 50, 5), [CLASSES[i] for i in range(10)], fontsize=7, rotation=45, ha='left')
ax.set_xlabel('selected concepts, grouped by the class they were selected for')
ax.set_title('LaBo: weight matrix A after training (row = class)')
fig.colorbar(im, ax=ax, fraction=0.02)
fig.tight_layout()
plt.show()

## 6. Comparison with the CBM from `PRACTICE_cbm_cem`

The CBM from the notebook `PRACTICE_cbm_cem` has two parts, `concept_encoder` and `task_predictor`, and learns from labeled concepts. CIFAR-10 has no concept labels, so we put binary labels from the same CLIP in their place: concept $j$ is “present” in an image if its cosine is above the 90th percentile of that concept over the training images. The concepts are the same as in the Label-free CBM bottleneck. The architecture is the one from that notebook with dimensions for 512 features and 10 classes; training is joint: BCE over concepts plus $0.2\times$ cross-entropy over the class, as in the first example of that notebook.

In [ ]:
class Concept_Bottleneck_Model(torch.nn.Module):   # as in PRACTICE_cbm_cem

    def __init__(self, input_size, hidden_size, concept_size, len_out):
        super().__init__()
        self.concept_encoder = torch.nn.Sequential(torch.nn.Linear(input_size, hidden_size), torch.nn.LeakyReLU(),
                                                   torch.nn.Linear(hidden_size, concept_size), torch.nn.Sigmoid())
        self.task_predictor = torch.nn.Sequential(torch.nn.Linear(concept_size, hidden_size), torch.nn.LeakyReLU(),
                                                  torch.nn.Linear(hidden_size, len_out))


T_lf = encode_text(lf_names)
P_train, P_test = X['train'] @ T_lf.T, X['test'] @ T_lf.T
q90 = P_train.quantile(0.9, dim=0)
L_train, L_test = (P_train > q90).float(), (P_test > q90).float()

torch.manual_seed(SEED)
cbm = Concept_Bottleneck_Model(512, 64, len(lf_names), 10).to(device)
opt = torch.optim.AdamW(cbm.parameters(), lr=1e-3)
for epoch in range(500):
    opt.zero_grad()
    c_pred = cbm.concept_encoder(B['train'])
    loss = F.binary_cross_entropy(c_pred, L_train) + 0.2 * F.cross_entropy(cbm.task_predictor(c_pred), y['train'])
    loss.backward()
    opt.step()
with torch.no_grad():
    c_pred = cbm.concept_encoder(B['test'])
    task_acc = (cbm.task_predictor(c_pred).argmax(1) == y['test']).float().mean().item()
    concept_acc = ((c_pred > 0.5) == L_test.bool()).float().mean().item()
results['CBM from PRACTICE_cbm_cem, concept labels from CLIP'] = (
    task_acc, sum(int((p != 0).sum()) for p in cbm.task_predictor.parameters() if p.dim() == 2))
print(f'concept accuracy {concept_acc:.4f} with a share of zeros in the labels of {1 - L_test.mean().item():.4f}')

In [ ]:
table = pd.DataFrame([{'model': k, 'test accuracy': round(a, 4), 'nonzero last-layer weights': n}
                      for k, (a, n) in results.items()])
table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
colors = ['#7f8c8d', '#7f8c8d', '#7f8c8d', '#2a6f97', '#2a6f97', '#e67e22', '#8e44ad']
labels_en = ['CLIP zero-shot', 'linear over ResNet-18', 'linear over CLIP', 'Label-free CBM, λ=0',
             f'Label-free CBM, λ={LAM}', 'LaBo, k=5', 'CBM (CLIP concept labels)']
for (k, (a, n)), c, lab in zip(results.items(), colors, labels_en):
    if n > 0:
        ax.scatter(n, a, color=c, s=50)
        ax.annotate(lab, (n, a), textcoords='offset points', xytext=(6, -3), fontsize=8)
ax.axhline(results['CLIP zero-shot'][0], color='#7f8c8d', ls='--', lw=1)
ax.text(25, results['CLIP zero-shot'][0] + 0.004, 'CLIP zero-shot (no trained weights)', fontsize=8, color='#7f8c8d')
ax.set_xscale('log')
ax.set_xlim(20, 20000)
ax.set_xlabel('nonzero weights of the last layer (log scale)')
ax.set_ylabel('test accuracy')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**Task 6. From the table and the plot: which concept method gives the last layer with the fewest nonzero weights, how many accuracy points does it lose to the linear classifier over the features of the same ResNet-18 backbone, and why can LaBo be compared in accuracy only with CLIP and not with ResNet-18?**

**Your answer:**

## 7. Where it breaks

**The class name in the bottleneck.** We remove the class-name similarity filter (`class_cut=1.01` switches it off) and build Label-free CBM again. If `a cat`, `aircraft`, `bird` stay in the list, the strongest concept of a class can turn out to be just its name: accuracy stays the same, and the explanation “it is a cat because there is a cat in the image” explains nothing. Look also at the column “with the filter”: the filter let `a cat bed`, `a deer stand`, `a dog bowl` through because their cosine with the class name is below 0.90, yet the class name is still in them. A cosine filter catches near-exact copies of the name, not concepts that contain the name.

In [ ]:
leak_names, C_leak, _ = concept_layer(presence_filter(text_filters(concepts, class_cut=1.01), verbose=False), verbose=False)
W_leak, b_leak = fit_sparse(C_leak['train'], y['train'], LAM)
print(f'concepts {len(leak_names)}, accuracy {accuracy(C_leak["test"], W_leak, b_leak, y["test"]):.4f}, nonzero weights {int((W_leak != 0).sum())}')
pd.DataFrame({'with the filter': [lf_names[W_lf[i].argmax()] for i in range(10)],
              'without the filter': [leak_names[W_leak[i].argmax()] for i in range(10)]}, index=CLASSES)

**What CLIP does not see in a 32×32 image.** The presence filter in section 2 threw out `headlights`, `fur`, `a steering wheel`, `side windows`: GPT-3 described the class as in a high-resolution photo, while in a CIFAR-10 image these details take a few pixels, and the CLIP cosine with them does not rise above 0.25 on any training image. A concept list from a language model does not know at what resolution the data will be.

**The weight moves off the language prior.** In section 5, at the start of LaBo training about 0.94 of the row weight lay on the class's own concepts, and after training in the solution run seven classes of ten keep less than half (from 0.29 for `automobile` to 0.48 for `deer`, see the output of section 5): the classifier takes any concepts that help, and on the heat map every row has weight on concepts selected for other classes. The LaBo explanation is read from the matrix $A$, not from the initial selection.

**A concept's value is the similarity CLIP computes.** The picture in section 4 shows on which images a concept is most active; whether they match what an annotator would name is checked only by eye. The concept accuracy of the CBM in section 6 is close to the share of zeros in the labels: binary labels from CLIP are rare, and predicting “no” almost everywhere already gives high concept accuracy. High concept accuracy without human labels says nothing about whether the concepts are right.

## What to take away

- **Label-free CBM and LaBo replace concept labels with two ready-made models.** A language model writes the list, CLIP estimates how visible a concept is in an image. Label-free CBM trains a projection of backbone features to reproduce these estimates, LaBo uses them directly as the bottleneck.
- **The concept filters are part of the method, not cleanup.** Without the class-name similarity filter the bottleneck passes the name itself, and the explanation degenerates; a cosine filter still misses concepts with the name inside (`a deer stand`).
- **A sparse last layer is shorter and almost as accurate.** Elastic net with a proximal step at $\lambda=0.003$ leaves about two dozen of about a hundred concepts per class, and test accuracy drops by about 2 points; accuracy and the number of nonzero weights for other $\lambda$ are in the table of section 4.
- **In LaBo the prior only sets the start of training.** After training the weight moves to other classes' concepts, and the explanation has to be read from the trained matrix.
- **Without human labels the quality of concepts cannot be measured.** Both concept accuracy and the pictures of the most active images tell which similarity to the concept CLIP computes.

Sources:
1. [Oikarinen et al., 2023. Label-Free Concept Bottleneck Models](https://arxiv.org/abs/2304.06129), code: [Trustworthy-ML-Lab/Label-free-CBM](https://github.com/Trustworthy-ML-Lab/Label-free-CBM)
2. [Yang et al., 2023. Language in a Bottle: Language Model Guided Concept Bottlenecks for Interpretable Image Classification](https://arxiv.org/abs/2211.11158), code: [YueYANG1996/LaBo](https://github.com/YueYANG1996/LaBo)
3. [Koh et al., 2020. Concept Bottleneck Models](https://arxiv.org/abs/2007.04612)
4. [Radford et al., 2021. Learning Transferable Visual Models From Natural Language Supervision (CLIP)](https://arxiv.org/abs/2103.00020)